# Módulo 03: Joins Optimizados, Window Functions y Lógica Modular

## 1. Estrategias de Unión: Sort-Merge Join vs. Broadcast Hash Join

En sistemas distribuidos, unir dos tablas puede ser la operación más costosa de un pipeline debido al movimiento de datos a través de la red (*Shuffle*).

### Sort-Merge Join (Enfoque por defecto)
Cuando dos tablas de gran volumen se unen sin optimizaciones específicas:
1. **Shuffle Phase:** Spark redistribuye las filas de ambas tablas por la red usando el hash de la clave de unión, agrupando las mismas claves en el mismo ejecutor.
2. **Sort Phase:** Cada partición se ordena en disco y memoria.
3. **Merge Phase:** Se fusionan los registros ordenados.
*Costo:* Alto consumo de I/O de red, uso intensivo de disco y riesgo de cuellos de botella por particiones desbalanceadas (*Data Skew*).

### Broadcast Hash Join (`F.broadcast`)
Si una de las dos tablas es pequeña (dimensiones, catálogos, tablas de referencia):
1. El Driver recolecta la tabla pequeña y envía una copia idéntica a la memoria de cada ejecutor (*Broadcast*).
2. La tabla grande no se mueve por la red; se procesa localmente en cada nodo comparándola contra la copia en memoria.
*Costo:* Cero shuffle en la tabla grande, reduciendo tiempos de ejecución drásticamente.

In [15]:
import sys
import os

# Permitir importaciones relativas desde src/
sys.path.append(os.path.abspath(".."))

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

from src.olympics_pipeline import (
    DEPORTISTAS_SCHEMA,
    EQUIPOS_SCHEMA,
    calcular_imc,
    clasificar_imc_categoria
)

# Inicialización canónica
spark = (
    SparkSession.builder
    .appName("03_Transformaciones_y_SQL")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

# Cargar datasets con contratos estrictos
df_deportistas = (
    spark.read
    .schema(DEPORTISTAS_SCHEMA)
    .option("header", "true")
    .csv("../data/raw/deportista.csv")
)

df_equipos = (
    spark.read
    .schema(EQUIPOS_SCHEMA)
    .option("header", "true")
    .csv("../data/raw/equipos.csv")
)

print(f"Deportistas cargados: {df_deportistas.count()}")
print(f"Equipos cargados: {df_equipos.count()}")

Deportistas cargados: 12
Equipos cargados: 8


## 2. Aplicación de Broadcast Join y Reutilización Modular

Al aplicar `F.broadcast(df_equipos)`, le indicamos al optimizador Catalyst que omita la fase de shuffle para la tabla dimensional.

Posteriormente, reutilizamos las funciones puras importadas desde `src/olympics_pipeline.py`, garantizando que la misma lógica evaluada en las pruebas unitarias sea la que se ejecuta en los cuadernos.

In [16]:
# 1. Broadcast Hash Join explícito
df_enriquecido = df_deportistas.join(
    F.broadcast(df_equipos),
    on="equipo_id",
    how="inner"
)

# 2. Aplicación encadenada de pipelines modulares
df_completo = (
    df_enriquecido
    .transform(calcular_imc)
    .transform(clasificar_imc_categoria)
)

df_completo.select(
    "deportista_id", "nombre", "equipo", "altura", "peso", "imc", "categoria_imc"
).show(5)

# 3. Inspección del plan físico para verificar el operador BroadcastHashJoin
df_completo.explain(mode="simple")

+-------------+--------------------+--------------+------+----+-----+-------------+
|deportista_id|              nombre|        equipo|altura|peso|  imc|categoria_imc|
+-------------+--------------------+--------------+------+----+-----+-------------+
|            1|           A Dijiang|         China| 180.0|80.0|24.69|       Normal|
|            2|            A Lamusi|       China-2| 170.0|60.0|20.76|       Normal|
|            3|      Gunnar Nielsen|       Denmark| 185.0|82.0|23.96|       Normal|
|            4|Edgar Lindenau Aabye|Denmark/Sweden| 182.0|81.0|24.45|       Normal|
|            5|Christine Jacoba ...|   Netherlands| 185.0|72.0|21.04|       Normal|
+-------------+--------------------+--------------+------+----+-----+-------------+
only showing top 5 rows

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [equipo_id#822, deportista_id#816, nombre#817, genero#818, edad#819, altura#820, peso#821, equipo#831, sigla#832, imc#867, CASE WHEN (imc#867 < 18.5) TH

## 3. Funciones Analíticas de Ventana (Window Functions)

A diferencia de las agregaciones con `groupBy()` (que colapsan múltiples filas en una sola), las **Window Functions** calculan métricas acumuladas, particionadas o de ranking **manteniendo la granularidad original de cada fila**.

Una especificación de ventana (`WindowSpec`) se compone de:
* **`partitionBy()`**: Segmenta los datos en grupos independientes.
* **`orderBy()`**: Establece la secuencia interna de evaluación dentro de cada partición.
* **`rowsBetween()` / `rangeBetween()`**: Delimita el marco móvil de cálculo (opcional).

### Funciones de ranking principales:
* `F.row_number()`: Asigna un índice consecutivo estricto sin empates ($1, 2, 3, 4$).
* `F.rank()`: Deja huecos cuando detecta valores idénticos ($1, 2, 2, 4$).
* `F.dense_rank()`: No deja huecos ante valores idénticos ($1, 2, 2, 3$).

In [17]:
# Definición de la ventana: particionada por equipo y ordenada por altura descendente
ventana_altura = (
    Window
    .partitionBy("equipo")
    .orderBy(F.col("altura").desc_nulls_last())
)

# Cálculo de rankings relativos dentro de cada país
df_ranking = df_completo.withColumn(
    "ranking_altura_equipo",
    F.dense_rank().over(ventana_altura)
)

df_ranking.show(10)

# Filtrar al atleta más alto de cada equipo
df_lideres_altura = (
    df_ranking
    .filter(F.col("ranking_altura_equipo") == 1)
    .select("equipo", "nombre", "altura", "ranking_altura_equipo")
    .orderBy(F.col("altura").desc())
)

df_lideres_altura.show(10)

+---------+-------------+--------------------+------+----+------+----+--------------+-----+-----+-------------+---------------------+
|equipo_id|deportista_id|              nombre|genero|edad|altura|peso|        equipo|sigla|  imc|categoria_imc|ranking_altura_equipo|
+---------+-------------+--------------------+------+----+------+----+--------------+-----+-----+-------------+---------------------+
|        1|            1|           A Dijiang|     1|  24| 180.0|80.0|         China|  CHN|24.69|       Normal|                    1|
|        2|            2|            A Lamusi|     1|  23| 170.0|60.0|       China-2|  CHN|20.76|       Normal|                    1|
|        3|            3|      Gunnar Nielsen|     1|  24| 185.0|82.0|       Denmark|  DEN|23.96|       Normal|                    1|
|        4|            4|Edgar Lindenau Aabye|     1|  34| 182.0|81.0|Denmark/Sweden|  DEN|24.45|       Normal|                    1|
|        7|            9|    Antti Sami Aalto|     1|  26| 186

## 4. Reto Práctico: Identificación del Atleta Más Veterano por Equipo

### Instrucciones del Ejercicio:
1. Define una especificación de ventana particionada por `"equipo"` y ordenada por `"edad"` de forma descendente (`desc()`).
2. Agrega una columna llamada `"ranking_edad"` utilizando `F.row_number()`.
3. Filtra el DataFrame para conservar únicamente al atleta más veterano de cada equipo (`ranking_edad == 1`).
4. Extrae los datos del representante de `"Finland"` y valida con la aserción automática.

In [18]:
# 1. Definición de la ventana analítica
ventana_edad = (
    Window
    .partitionBy("equipo")
    .orderBy(F.col("edad").desc())
)

# 2. Generación del índice de posición
df_con_ranking_edad = df_completo.withColumn(
    "ranking_edad",
    F.row_number().over(ventana_edad)
)

# 3. Filtrado de atletas con mayor edad por país
df_veteranos = (
    df_con_ranking_edad
    .filter(F.col("ranking_edad") == 1)
    .select("equipo", "nombre", "edad")
)

df_veteranos.show()

# 4. Extracción de los datos de Finland
veterano_finland = (
    df_veteranos
    .filter(F.col("equipo") == "Finland")
    .collect()[0]
)

nombre_obtenido = veterano_finland["nombre"]
edad_obtenida = veterano_finland["edad"]

print(f"Atleta más veterano de Finland: {nombre_obtenido} ({edad_obtenida} años)")

# 5. Validación automática del reto
assert nombre_obtenido == "Jyri Tapani", f"Esperado 'Jyri Tapani', obtenido '{nombre_obtenido}'"
assert edad_obtenida == 27, f"Esperada edad 27, obtenida {edad_obtenida}"

print("¡Aserción aprobada! Cuaderno 03 completado con éxito.")

+--------------+--------------------+----+
|        equipo|              nombre|edad|
+--------------+--------------------+----+
|         China|           A Dijiang|  24|
|       China-2|            A Lamusi|  23|
|       Denmark|      Gunnar Nielsen|  24|
|Denmark/Sweden|Edgar Lindenau Aabye|  34|
|       Finland|         Jyri Tapani|  27|
|   Netherlands|Christine Jacoba ...|  21|
|        Norway|Einar Ferdinand U...|  20|
| United States|     Per Knut Aaland|  31|
+--------------+--------------------+----+

Atleta más veterano de Finland: Jyri Tapani (27 años)
¡Aserción aprobada! Cuaderno 03 completado con éxito.
